# آموزش full-archive **v2** روی Colab — patch store (۴۹۰ مزرعه / ۳٬۴۷۸ سایکل معتبر)

همان ۶ آموزش دست‌نخوردهٔ handover روی GPU کولب: `patch_pixelwise` و `patch_cnn_gru`
× seeds `13, 42, 2026` — ۱۰ epoch — این بار روی **استور v2** (جدول سایکل fused:
۳٬۴۷۸ سایکل معتبر به‌جای ۷۹۹ در v1؛ v1 دست‌نخورده می‌ماند و مقایسهٔ v1↔v2 خودش یک deliverable است).

**قبل از اجرا:**
1. Runtime → Change runtime type → **T4 GPU**
2. ریپو public و push شده باشد: `Mahdikalantari555/thesis_implementation` — نتیجهٔ مورد انتظار: commit `67b8f82` یا جدیدتر
3. فایل `thesis_data_full_v2.zip` (~۲۹۴ MB، ۱۲٬۹۰۳ فایل) را در Drive در مسیر `MyDrive/thesis_colab/` بگذارید

`--resume` فعال است؛ اگر session قطع شد فقط سلول آموزش را دوباره اجرا کنید.
داده ۴٫۴× برابر v1 است؛ تخمین کل ۶ آموزش روی T4: حدود **۲ تا ۵ ساعت** — در صورت قطع شدن، در چند session ادامه دهید.

In [ ]:
import subprocess, sys, torch

assert torch.cuda.is_available(), (
    "GPU فعال نیست — Runtime > Change runtime type > T4 GPU را انتخاب کنید"
)
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "zarr==3.3.0", "lightning", "click"], check=True)
import zarr
print("zarr:", zarr.__version__)

In [ ]:
import subprocess

REPO_URL = "https://github.com/Mahdikalantari555/thesis_implementation.git"
r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL,
                    "/content/thesis_implementation"])
if r.returncode != 0:
    raise SystemExit("clone ناموفق — مطمئن شوید ریپو public و push شده است")
head = subprocess.run(["git", "-C", "/content/thesis_implementation", "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
print("*(handover v2 ثبت‌شده روی commit 67b8f82 است؛ HEAD جدید یعنی commitهای تازه‌تر)*")

In [ ]:
from google.colab import drive
import pathlib, shutil, zipfile

drive.mount("/content/drive")
ZIP = pathlib.Path("/content/drive/MyDrive/thesis_colab/thesis_data_full_v2.zip")
assert ZIP.is_file(), f"زیپ دیتای v2 پیدا نشد — آن را در {ZIP} بگذارید"
shutil.copy(ZIP, "/content/data_v2.zip")
with zipfile.ZipFile("/content/data_v2.zip") as z:
    z.extractall("/content/thesis_implementation/")
    print("extracted:", len(z.namelist()), "entries into repo data/")

In [ ]:
import os, sys, json

os.chdir("/content/thesis_implementation")
sys.path.insert(0, "/content/thesis_implementation/src")

import zarr
from ssl_anomaly.io import patches as P
from ssl_anomaly.monitor import splits as S
from ssl_anomaly.monitor import holdout as H

STORE = "data/patch_runs/v3/patches.zarr"
SPLITS = "data/processed/splits_locked_v2.json"
HOLDOUT = "data/processed/holdout_manifest_v2.json"
NORM = "data/processed/norm_stats_full_v2.json"

man = S.load_splits_locked(path=SPLITS, verify_cycle_files=False)
hold = H.load_holdout_manifest(path=HOLDOUT)
root = zarr.open_group(STORE, mode="r")
P.validate_store(root, expected_fields=490)

prov = json.loads(open(f"{STORE}/provenance.json").read())
assert prov["splits_manifest_sha256"] == man["manifest_sha256"], (
    "استور v2 با تقسیم‌های قفل‌شدهٔ v2 هم‌خوان نیست — بستهٔ دیتا را دوباره بسازید")
assert prov["cycle_source_sha256"] == man["cycle_file_hashes"]["cycles_parquet_sha256"], (
    "جدول سایکل استور با splits v2 فرق دارد")
assert root["values"].shape == (490, 12, 36, 8, 48, 48), root["values"].shape
assert int(root["cycle_valid"][:].sum()) == 3478, int(root["cycle_valid"][:].sum())

norm = P.load_patch_norm(NORM)  # self-hash داخل خودش verify می‌شود
assert norm["splits_sha256"] == man["manifest_sha256"], "نرمال‌سازی با splits v2 بایند نیست"
assert norm["holdout_sha256"] == H.holdout_manifest_hash(hold), "نرمال‌سازی با holdout v2 بایند نیست"
assert norm["n_train_cycles"] == 2105, norm["n_train_cycles"]

for name in S.SPLIT_NAMES:
    print(f"{name}: {len(man['cycles'].get(name, []))} cycles")
print("gates OK — v2 store bound to splits", man["manifest_sha256"][:12])

In [ ]:
import os, subprocess, sys, time

os.chdir("/content/thesis_implementation")
env = dict(os.environ, PYTHONPATH="/content/thesis_implementation/src")
for arch in ["patch_pixelwise", "patch_cnn_gru"]:
    for seed in [13, 42, 2026]:
        rdir = f"checkpoints/patch/patch_full_v2_ep10/{arch}_seed{seed}"
        extra = ["--resume"] if os.path.exists(f"{rdir}/last.pt") else []
        print(f"=== {arch} seed={seed}  start: {time.strftime('%H:%M:%S')} ===",
              flush=True)
        r = subprocess.run(
            [sys.executable, "-X", "utf8", "-u", "scripts/train_patch.py",
             "--arch", arch, "--seed", str(seed), "--epochs", "10",
             "--store", "data/patch_runs/v3/patches.zarr",
             "--splits", "data/processed/splits_locked_v2.json",
             "--holdout-manifest", "data/processed/holdout_manifest_v2.json",
             "--norm", "data/processed/norm_stats_full_v2.json",
             "--out", "checkpoints/patch/patch_full_v2_ep10", *extra],
            env=env,
        )
        if r.returncode != 0:
            print(f"FAILED: {arch} seed={seed} (exit {r.returncode})", flush=True)
print(f"=== ALL DONE: {time.strftime('%H:%M:%S')} ===")

In [ ]:
import os, shutil

os.chdir("/content/thesis_implementation")
base = shutil.make_archive("/content/patch_full_v2_ep10_results", "zip",
                           root_dir="checkpoints/patch", base_dir="patch_full_v2_ep10")
dest = "/content/drive/MyDrive/thesis_colab/patch_full_v2_ep10_colab_results.zip"
shutil.copy(base, dest)
print("results copied to Drive:", dest,
      f"({os.path.getsize(dest)/1e6:.1f} MB)")

## بعد از اتمام

* خروجی‌ها در Drive: `MyDrive/thesis_colab/patch_full_v2_ep10_colab_results.zip`
  (۶ پوشهٔ run، هر کدام: ۱۰ `epochNNN.pt` + `last.pt` + `norm_stats.json` + `progress.jsonl` + `run_manifest.json`)
* نکتهٔ reproducibility: seedها ثابت‌اند ولی GPU در مقابل CPU ممکن است اختلاف float جزئی بدهد؛ دستگاه و wall time داخل `run_manifest.json` هر run ثبت شده است.
* گام بعد: zip نتایج را در `D:/RS/Thesis/thesis_implementation/checkpoints/patch/` باز کنید تا sigma ×۶ و eval شدت ×۶ روی v2 اجرا شود و جدول مقایسهٔ v1↔v2 ساخته شود.